In [86]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [87]:
data = pd.read_csv("../../datasets/titanic/train.csv")

In [88]:
data.head(5)

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [89]:
data.isnull().sum().sum()

866

In [90]:
missing_stats = pd.DataFrame({
    "missing_count": data.isnull().sum(),
    "missing_%": data.isnull().mean() * 100
})

missing_stats = missing_stats.sort_values(by="missing_%", ascending=False)
missing_stats = missing_stats[missing_stats["missing_count"] > 0]
print(missing_stats)

          missing_count  missing_%
Cabin               687  77.104377
Age                 177  19.865320
Embarked              2   0.224467


In [91]:
data["Cabin"] = data["Cabin"].fillna("NA")

In [92]:
data["Age"] = data["Age"].fillna(data["Age"].mean())

In [93]:
data[data["Embarked"].isna()].index

Index([61, 829], dtype='int64')

In [94]:
data = data.drop(index=[61, 829])

In [95]:
data.duplicated().sum()

0

In [96]:
data = data.drop(columns=['Ticket', 'Cabin', 'Name'])

In [97]:
num_cols = data.select_dtypes(include = "number").columns

Q1 = data[num_cols].quantile(0.25)
Q3 = data[num_cols].quantile(0.75)

IQR = Q3-Q1

outliers = (data[num_cols] < (Q1 - 1.5*IQR)) | (data[num_cols] > (Q3 + 1.5*IQR))
print(outliers)

     PassengerId  Survived  Pclass    Age  SibSp  Parch   Fare
0          False     False   False  False  False  False  False
1          False     False   False  False  False  False   True
2          False     False   False  False  False  False  False
3          False     False   False  False  False  False  False
4          False     False   False  False  False  False  False
..           ...       ...     ...    ...    ...    ...    ...
886        False     False   False  False  False  False  False
887        False     False   False  False  False  False  False
888        False     False   False  False  False   True  False
889        False     False   False  False  False  False  False
890        False     False   False  False  False  False  False

[889 rows x 7 columns]


In [98]:
cat_cols = data.select_dtypes(include = "object").columns

In [99]:
data = pd.get_dummies(data, columns = cat_cols, drop_first = True)

In [100]:
X = data.drop("Survived", axis = 1)
y = data["Survived"]

In [101]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

In [102]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size = 0.2, random_state = 16)

In [103]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(solver='saga', max_iter=1000)
model.fit(X_train, y_train)

predictions = model.predict(X_test)

In [104]:
from sklearn.metrics import accuracy_score

accuracy = accuracy_score(y_test, predictions)
print(accuracy)

0.8146067415730337


In [105]:
#random forest
from sklearn.ensemble import RandomForestClassifier

model = RandomForestClassifier(n_estimators = 2000, max_depth = 25, min_samples_leaf = 30, min_samples_split = 20, n_jobs = -1, max_features = "sqrt", random_state = 16)
model.fit(X_train, y_train)

importance = pd.Series(model.feature_importances_, index = X.columns).sort_values(ascending = False)

predictions = model.predict(X_test)

In [106]:
from sklearn.metrics import classification_report

print("Accuracy:", accuracy_score(y_test, predictions))
print(classification_report(y_test, predictions))

Accuracy: 0.8146067415730337
              precision    recall  f1-score   support

           0       0.79      0.93      0.85       102
           1       0.88      0.66      0.75        76

    accuracy                           0.81       178
   macro avg       0.83      0.79      0.80       178
weighted avg       0.82      0.81      0.81       178



In [107]:
#XGBoost
from xgboost import XGBClassifier

model = XGBClassifier(n_estimators=2000, max_depth=4, learning_rate=0.08, subsample=0.8, colsample_bytree=0.8, random_state=16, eval_metric='logloss')
model.fit(X_train, y_train)

predictions = model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, predictions))
print(classification_report(y_test, predictions))

Accuracy: 0.8258426966292135
              precision    recall  f1-score   support

           0       0.83      0.87      0.85       102
           1       0.82      0.76      0.79        76

    accuracy                           0.83       178
   macro avg       0.82      0.82      0.82       178
weighted avg       0.83      0.83      0.82       178

